In [1]:
#Effective energies of the Delta smearing tuning (40 cfgs, 24 operators, 32 sources), pion and proton, one page per pz:
#  meff_<particle>_<gamma>.pdf      diagonal correlator of each operator, gamma = G5, G5G4, G3G5 at sink and source
#  meff_<particle>_GEVP_...pdf      reconstructed 2pt of the 3x3 GEVP over the gammas of each operator
#  ratio_<particle>_GEVP_...pdf     [C_rec(t) / C_rec(t_norm)] of each operator divided by the same ratio of the reference operator
#C = (forward + s_a s_b backward) / 2 with s = +1, -1, +1 for G5, G5G4, G3G5 (tsep = 0: forward only, the proton backward source slice is projected
#with parity_m); the t_src are averaged in each cfg, errors are jackknife over the cfgs. E_eff(t) = log(C(t) / C(t+1)).
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

data_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning"
file_name = "2pt_coulomb_tune_32srcs_40cfgs_24operators_24ops_{particle}_tsrc_fb_tsep20_32src_ncfg40.h5"
plot_dir = f"{data_dir}/plots"
os.makedirs(plot_dir, exist_ok=True)

particles = ["pion", "proton"]
t0 = 2                                  # GEVP: C(t) v = lambda C(t0) v
t_vec = 4                               # eigenvectors at t_vec from the full average, kept fixed for the jackknife samples
t_norm = 3                              # ratio plot: C_rec(t) / C_rec(t_norm)
ref_row = [3.25, 3.25 / 2, 0.7]         # ratio plot: every operator divided by this one
t_max_plot = 16
err_max = 0.1                           # meff points with a larger jackknife error are not drawn (None: all drawn)
ratio_err_max = 0.05                    # the same for the ratio plot; these points do not set its y range either
m_disp = {"pion": 0.141, "proton": None}                   # am of the dispersion line (pion: compare_3rhoz_pion.py); None: no line
ylim_window = {"pion": (0.05, 0.3), "proton": (0.1, 0.5)}   # meff y range: center - lower, center + upper (center: dispersion, or the GEVP plateau)
Ls = 32
gamma_sign = {"G5": 1, "G5G4": -1, "G3G5": 1}             # time reflection C_ab(-t) = s_a s_b C_ab(t), as in the h5 attribute time_reflection


def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err

def meff(c):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.log(c[..., :-1] / c[..., 1:])

def as_str(x):
    return x.decode() if isinstance(x, bytes) else str(x)


def load(particle):
    with h5py.File(f"{data_dir}/{file_name.format(particle=particle)}", "r") as f:
        assert as_str(f.attrs["dim_pt2"]) == "cfg, smear, gamma_sink, gamma_source, t_src, pf, tsep", f.attrs["dim_pt2"]
        pt2_forward = f["pt2_forward"][:].mean(axis=4)              # (cfg, smear, g_snk, g_src, pf, tsep), t_src averaged
        pt2_backward = f["pt2_backward"][:].mean(axis=4)
        pf_list = f["pf_list"][:]
        smear_list = f["smear_list"][:]
        gamma_list = [as_str(g) for g in f.attrs["gamma_list"]]
    s = np.array([gamma_sign[g] for g in gamma_list])
    C = 0.5 * (pt2_forward + s[None, None, :, None, None, None] * s[None, None, None, :, None, None] * pt2_backward)
    C[..., 0] = pt2_forward[..., 0]
    return C, pf_list, smear_list, gamma_list


def gevp_rec(C_mean, C_jk):
    #per operator and pz: 3x3 GEVP over the gammas, ground-state eigenvector v0 at (t_vec, t0) normalized to v0^dag C(t0) v0 = 1,
    #reconstructed 2pt v0^dag C(t) v0 of the full average and of every jackknife sample (same v0).
    #C(t0) = U w U^dag; L_inv = w^-1/2 U^dag on the directions with w > 0, so that L_inv C(t0) L_inv^dag = 1: the same as the Cholesky
    #L^-1 when C(t0) is positive definite, and the GEVP in the positive directions when noise makes an eigenvalue of C(t0) negative
    n_cfg, n_smear, _, _, n_pf, n_tsep = C_jk.shape
    rec_mean = np.full((n_smear, n_pf, n_tsep), np.nan)
    rec_jk = np.full((n_cfg, n_smear, n_pf, n_tsep), np.nan)
    for i in range(n_smear):
        for p in range(n_pf):
            A = C_mean[i, :, :, p]                                          # (g_snk, g_src, tsep)
            A = 0.5 * (A + np.conj(np.swapaxes(A, 0, 1)))                   # hermitian: the G5-G3G5 and G5G4-G3G5 elements are imaginary
            sign = np.sign(np.trace(A[:, :, t0]).real)                      # the pion comes out negative in this contraction convention
            w0, u0 = np.linalg.eigh(sign * A[:, :, t0])
            keep = w0 > 1e-6 * w0.max()
            if not keep.all():
                print(f"  GEVP operator {i}, pz index {p}: C(t0) eigenvalues {np.round(w0 / w0.max(), 6)}, GEVP in the {keep.sum()} positive directions")
            L_inv = (u0[:, keep] / np.sqrt(w0[keep])).conj().T
            w, u = np.linalg.eigh(L_inv @ (sign * A[:, :, t_vec]) @ L_inv.conj().T)
            v0 = L_inv.conj().T @ u[:, np.argmax(w)]
            B = C_jk[:, i, :, :, p]                                         # (cfg, g_snk, g_src, tsep)
            B = 0.5 * (B + np.conj(np.swapaxes(B, 1, 2)))
            rec_mean[i, p] = sign * np.einsum("a,abT,b->T", np.conj(v0), A, v0).real
            rec_jk[:, i, p] = sign * np.einsum("a,cabT,b->cT", np.conj(v0), B, v0).real
    return rec_mean, rec_jk


def styles(smear_list, i_ref):
    #color: rho_T; marker: (rho_T / rho_z, mom_frac); open markers for rho_z = rho_T/3
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown", "tab:pink", "tab:olive", "tab:cyan", "black"]
    markers = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]
    rho_T_all = sorted(set(np.round(smear_list[:, 0], 6)))
    shape_all = sorted(set((round(rT / rz, 3), round(fr, 6)) for rT, rz, fr in smear_list))
    out = []
    for i, (rT, rz, fr) in enumerate(smear_list):
        ratio = round(rT / rz, 3)
        rz_name = "rho_T" if ratio == 1 else f"rho_T/{ratio:g}"
        label = f"{i:2d}: rho_T {rT:g}, rho_z = {rz_name}, frac {fr:g}" + ("  (ref)" if i == i_ref else "")
        out.append((colors[rho_T_all.index(round(rT, 6)) % 10], markers[shape_all.index((ratio, round(fr, 6))) % 10], ratio < 3, label))
    return out


def page(pdf, t, y, e, style, title, ylabel, ylim, hline=None, hline_label=None, e_cut=None):
    fig, ax = plt.subplots(figsize=(28, 16), constrained_layout=True)
    n_op = y.shape[0]
    for i in range(n_op):
        color, marker, filled, label = style[i]
        ok = np.isfinite(y[i]) & np.isfinite(e[i]) & (t >= 1) & (t <= t_max_plot)
        if e_cut is not None:
            ok &= e[i] < e_cut
        ax.errorbar(t[ok] + 0.025 * (i - (n_op - 1) / 2), y[i][ok], e[i][ok], fmt=marker + "-", color=color, ms=9, mfc=color if filled else "none",
                    mew=1.5, lw=1.0, elinewidth=1.3, capsize=3, label=label)
    if hline is not None:
        ax.axhline(hline, color="gray", lw=1.5, ls="--", label=hline_label)
    ax.set_xlim(0.5, t_max_plot + 0.5)
    if np.all(np.isfinite(ylim)):
        ax.set_ylim(ylim)
    ax.set_xticks(np.arange(1, t_max_plot + 1))
    ax.tick_params(labelsize=18)
    ax.grid(alpha=0.3)
    ax.set_xlabel("t", fontsize=22)
    ax.set_ylabel(ylabel, fontsize=22)
    ax.set_title(title, fontsize=24)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=15, frameon=False)
    pdf.savefig(fig)
    plt.close(fig)


for particle in particles:
    C, pf_list, smear_list, gamma_list = load(particle)
    n_cfg, n_smear, _, _, n_pf, n_tsep = C.shape
    C_mean = C.mean(axis=0)
    C_jk = (C.sum(axis=0)[None] - C) / (n_cfg - 1)
    i_ref = int(np.where(np.isclose(smear_list, ref_row).all(axis=1))[0][0])
    style = styles(smear_list, i_ref)
    print(f"{particle}: {n_cfg} cfgs, {n_smear} operators, gammas {gamma_list}, pz {pf_list[:, 2].tolist()}, reference operator {i_ref} = {smear_list[i_ref]}")
    t_eff = np.arange(n_tsep - 1)
    t_all = np.arange(n_tsep)
    cut_note = f"   (points with jk error > {err_max} not drawn)" if err_max is not None else ""

    #GEVP over the gammas, reconstructed 2pt
    rec_mean, rec_jk = gevp_rec(C_mean, C_jk)
    meff_rec = meff(rec_mean)                                               # (smear, pf, t)

    #center of the meff y range: dispersion relation, else the GEVP plateau at t = 6..10 (reference operator, all operators, G5-G5)
    def center(p):
        if m_disp[particle] is not None:
            p_lat = 2 * np.pi * pf_list[p][2] / Ls
            return 2 * np.arcsinh(np.sqrt(np.sinh(m_disp[particle] / 2) ** 2 + np.sin(p_lat / 2) ** 2))
        for m in [meff_rec[i_ref, p], meff_rec[:, p], meff(C_mean[:, 0, 0].real)[:, p]]:
            if np.isfinite(m[..., 6:11]).any():
                return np.nanmedian(m[..., 6:11])
        return np.nan

    #meff of each gamma alone, then of the GEVP
    cases = [(g, C_mean[:, ig, ig].real, C_jk[:, :, ig, ig].real) for ig, g in enumerate(gamma_list)]
    for name, c_mean, c_jk in cases + [(f"GEVP_t0{t0}_tvec{t_vec}", rec_mean, rec_jk)]:
        y, e = meff(c_mean), jk(meff(c_jk))[1]
        what = f"{name}-{name}" if name in gamma_list else f"GEVP over {', '.join(gamma_list)} (t0 = {t0}, t_vec = {t_vec}), reconstructed 2pt"
        with PdfPages(f"{plot_dir}/meff_{particle}_{name}.pdf") as pdf:
            for p in range(n_pf):
                c = center(p)
                lo, hi = ylim_window[particle]
                ylim, note = (c - lo, c + hi), ""
                if name == "G3G5" and pf_list[p][2] == 0:                   # no ground-state overlap at rest: y range from the data
                    note = "   [pz = 0: no ground-state overlap" + (", P+ keeps only negative parity]" if particle == "proton" else "]")
                    v = y[:, p, 1:t_max_plot + 1][np.isfinite(y[:, p, 1:t_max_plot + 1]) & (e[:, p, 1:t_max_plot + 1] < (err_max or np.inf))]
                    if v.size:
                        ylim = (np.nanmin([ylim[0], v.min() - 0.05]), np.nanmax([ylim[1], v.max() + 0.05]))
                page(pdf, t_eff, y[:, p], e[:, p], style, f"{particle}  pz = {pf_list[p][2]}   {what}   {n_cfg} cfgs{cut_note}{note}",
                     "E_eff(t) = log C(t)/C(t+1)", ylim, c if m_disp[particle] is not None else None, f"dispersion, am = {m_disp[particle]}", err_max)
        print(f"  wrote {plot_dir}/meff_{particle}_{name}.pdf")

    #ratio plot: [C_rec(t) / C_rec(t_norm)] / [C_rec,ref(t) / C_rec,ref(t_norm)], each jackknife sample divided by its own reference
    with np.errstate(invalid="ignore", divide="ignore"):
        r_mean = rec_mean / rec_mean[:, :, t_norm:t_norm + 1]
        r_mean = r_mean / r_mean[i_ref][None]
        r_jk = rec_jk / rec_jk[:, :, :, t_norm:t_norm + 1]
        r_jk = r_jk / r_jk[:, i_ref][:, None]
    r_err = jk(r_jk)[1]
    with PdfPages(f"{plot_dir}/ratio_{particle}_GEVP_t0{t0}_tvec{t_vec}_tnorm{t_norm}.pdf") as pdf:
        for p in range(n_pf):
            vals, errs = r_mean[:, p, 1:t_max_plot + 1], r_err[:, p, 1:t_max_plot + 1]
            vals = vals[np.isfinite(vals) & (errs < ratio_err_max)]
            lo, hi = np.percentile(vals, [2, 98]) if vals.size else (np.nan, np.nan)
            pad = 0.1 * (hi - lo) + 1e-3
            page(pdf, t_all, r_mean[:, p], r_err[:, p], style,
                 f"{particle}  pz = {pf_list[p][2]}   [C_rec(t)/C_rec({t_norm})] / same for operator {i_ref}   GEVP over gammas, t0 = {t0}, t_vec = {t_vec}"
                 f"   (jk error > {ratio_err_max} not drawn)", "ratio to the reference operator", (lo - pad, hi + pad), 1.0, None, ratio_err_max)
    print(f"  wrote {plot_dir}/ratio_{particle}_GEVP_t0{t0}_tvec{t_vec}_tnorm{t_norm}.pdf")

pion: 40 cfgs, 24 operators, gammas ['G5', 'G5G4', 'G3G5'], pz [0, 1, 2, 3, 4, 5, 6], reference operator 9 = [3.25  1.625 0.7  ]
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_pion_G5.pdf
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_pion_G5G4.pdf
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_pion_G3G5.pdf
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_pion_GEVP_t02_tvec4.pdf
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/ratio_pion_GEVP_t02_tvec4_tnorm3.pdf
proton: 40 cfgs, 24 operators, gammas ['G5', 'G5G4', 'G3G5'], pz [0, 1, 2, 3, 4, 5, 6], reference operator 9 = [3.25  1.625 0.7  ]
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_proton_G5.pdf
  wrote /Users/mcp3270/Desktop/GLUON